# Monitors & Reports

This file sets up the model, data, and infrastructure monitors. It also establishes a monitoring dashboard the code for generating reports on SageMaker.

Attribution: The code was made with the assistance of Perplexity.ai accessed in February 2026.

### Set Up Model & Baseline

In [1]:
#Import key libraries

#Sagemaker Imports
import sagemaker
from sagemaker import Session
from sagemaker.model_monitor import (
    DefaultModelMonitor,
    ModelQualityMonitor,
    DatasetFormat,
    CronExpressionGenerator,
    EndpointInput
)
from sagemaker.model import Model
from sagemaker.predictor import Predictor
from sagemaker.serializers import IdentitySerializer
from sagemaker.deserializers import JSONDeserializer
from sagemaker import image_uris, get_execution_role


#Other Imports
import s3fs
import pandas as pd
import boto3
import json
import time
import tarfile
import botocore.exceptions
import io
from io import StringIO
from datetime import datetime, timedelta
import numpy as np
from pathlib import Path

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/sagemaker-user/.config/sagemaker/config.yaml


In [2]:
#Set up session
region = "us-east-1"
session = Session()
sm_client = boto3.client("sagemaker", region_name=region)
cw_client = boto3.client("cloudwatch", region_name=region)
s3_client = boto3.client("s3", region_name=region)

role = sagemaker.get_execution_role()
bucket = "sagemaker-us-east-1-418418308994"
prefix = "models/benchmarks"

In [3]:
# Config
bucket = "sagemaker-us-east-1-418418308994"
local_base = Path('/tmp/Models/benchmarks')
s3_base = f"s3://{bucket}/models/benchmarks"

print("Loading XGBoost artifacts...")

# XGBoost paths (prioritize local)
xgb_paths = {
    'local_tar.gz': local_base / 'xgboost/model.tar.gz',
    's3_tar.gz': f"{s3_base}/xgboost/model.tar.gz",
    's3_pkl': f"{s3_base}/xgboost/model.pkl",
    'local_metrics': local_base / 'xgboost/metrics.json',
    'local_model': local_base / 'xgboost/model.pkl'
}

# Auto-select best path
model_data = str(xgb_paths['local_tar.gz']) if xgb_paths['local_tar.gz'].exists() else xgb_paths['s3_tar.gz']
metrics_path = xgb_paths['local_metrics'] if xgb_paths['local_metrics'].exists() else None

print(f"Model data: {model_data}")
print(f"Metrics: {metrics_path}")
print("XGBoost paths ready!")

Loading XGBoost artifacts...
Model data: s3://sagemaker-us-east-1-418418308994/models/benchmarks/xgboost/model.tar.gz
Metrics: None
XGBoost paths ready!


In [4]:
# Get the latest XGBoost container for region
xgboost_container = image_uris.retrieve(
    framework="xgboost",
    region="us-east-1",
    version="1.7-1" 
)

print(f"Using XGBoost container: {xgboost_container}")

# Create model with updated container
new_xgb_endpoint_name = f"xgb-benchmark-endpoint-{datetime.now().strftime('%Y%m%d-%H%M%S')}"

xg_model_new = Model(
    image_uri=xgboost_container,
    model_data="s3://sagemaker-us-east-1-418418308994/models/benchmarks/xgboost/model.tar.gz",
    role=role,
    sagemaker_session=session,
)

print(f"Deploying new endpoint: {new_xgb_endpoint_name}")

# Deploy with data capture
data_capture_prefix = f"{prefix}/data-capture"
data_capture_s3_uri = f"s3://{bucket}/{data_capture_prefix}"

xg_predictor_new = xg_model_new.deploy(
    initial_instance_count=1,
    instance_type="ml.m5.large",
    endpoint_name=new_xgb_endpoint_name,
    data_capture_config=sagemaker.model_monitor.DataCaptureConfig(
        enable_capture=True,
        sampling_percentage=100,
        destination_s3_uri=data_capture_s3_uri,
        capture_options=["REQUEST", "RESPONSE"],
    ),
)

print(f"Endpoint deployed: {new_xgb_endpoint_name}")

Using XGBoost container: 683313688378.dkr.ecr.us-east-1.amazonaws.com/sagemaker-xgboost:1.7-1
Deploying new endpoint: xgb-benchmark-endpoint-20260210-045052
------!Endpoint deployed: xgb-benchmark-endpoint-20260210-045052


In [5]:
#Use if endpoint already deployed
#new_xgb_endpoint_name = "xgb-benchmark-endpoint-20260209-062303"

In [6]:
# Wait for endpoint to be in service
print("Waiting for endpoint to be ready...")

sm_client = boto3.client('sagemaker', region_name='us-east-1')

while True:
    response = sm_client.describe_endpoint(EndpointName=new_xgb_endpoint_name)
    status = response['EndpointStatus']
    print(f"  Status: {status}")
    
    if status == 'InService':
        print("Endpoint is ready!")
        break
    elif status == 'Failed':
        print("Endpoint deployment failed!")
        print(f"Failure reason: {response.get('FailureReason', 'Unknown')}")
        break
    
    time.sleep(30)

Waiting for endpoint to be ready...
  Status: InService
Endpoint is ready!


In [7]:
# Create predictor for the new endpoint
xg_predictor_new = Predictor(
    endpoint_name=new_xgb_endpoint_name,
    sagemaker_session=session,
)

# Load test data
baseline_df = pd.read_csv("s3://sagemaker-us-east-1-418418308994/models/benchmarks/baseline_normalized.csv")
test_data = baseline_df.drop("target", axis=1).iloc[:5]

print(f"Test data shape: {test_data.shape}")

# Convert to CSV
csv_payload = test_data.to_csv(header=False, index=False)

# Send prediction
response = xg_predictor_new.predict(
    data=csv_payload,
    initial_args={
        'ContentType': 'text/csv',
        'Accept': 'text/csv'
    }
)

print(f"✅ Predictions: {response}")

Test data shape: (5, 20)
✅ Predictions: b'0.009759249165654182,0.010805039666593075,0.48084768652915955,0.03209933266043663,0.03439685329794884,0.4277508556842804,0.004340957384556532\n0.007653615437448025,0.007332483772188425,0.22988218069076538,0.07651730626821518,0.07866891473531723,0.5871987342834473,0.012746804393827915\n0.01511878240853548,0.01036760676652193,0.5404502153396606,0.03245336189866066,0.12612764537334442,0.2628888487815857,0.012593579478561878\n0.015056170523166656,0.004233742132782936,0.17002418637275696,0.01482530776411295,0.09608134627342224,0.6938716769218445,0.005907570943236351\n0.02310902625322342,0.016121942549943924,0.2598908543586731,0.01709647849202156,0.08900441229343414,0.5792668461799622,0.015510435216128826\n'


In [8]:
try:
    dq_stats_uri = dq_monitor.latest_baselining_job.baseline_statistics.file_name
    dq_constraints_uri = dq_monitor.latest_baselining_job.suggested_constraints.file_name
    print(f"Stats: {dq_stats_uri}")
    print(f"Constraints: {dq_constraints_uri}")
except:
    # List S3 output manually if SDK fails
    s3 = boto3.client('s3')
    response = s3.list_objects_v2(Bucket=bucket, Prefix=f"{prefix}/monitoring/baseline/")
    for obj in response.get('Contents', []):
        if obj['Key'].endswith('statistics.json'):
            dq_stats_uri = f"s3://{bucket}/{obj['Key']}"
        if obj['Key'].endswith('constraints.json'):
            dq_constraints_uri = f"s3://{bucket}/{obj['Key']}"
    print(f"Stats: {dq_stats_uri}")
    print(f"Constraints: {dq_constraints_uri}")

Stats: s3://sagemaker-us-east-1-418418308994/models/benchmarks/monitoring/baseline/statistics.json
Constraints: s3://sagemaker-us-east-1-418418308994/models/benchmarks/monitoring/baseline/constraints.json


In [9]:
#Find JSON files
s3 = boto3.client('s3')

response = s3.list_objects_v2(Bucket=bucket, Prefix=f"{prefix}/monitoring/baseline/")
files = [obj['Key'] for obj in response.get('Contents', []) if obj['Key'].endswith('.json')]
print("JSON files:")
for f in files:
    print(f"s3://{bucket}/{f}")

JSON files:
s3://sagemaker-us-east-1-418418308994/models/benchmarks/monitoring/baseline/constraints.json
s3://sagemaker-us-east-1-418418308994/models/benchmarks/monitoring/baseline/statistics.json


In [10]:
#Check what constraints look like
key = "models/benchmarks/monitoring/baseline/constraints.json"
obj = s3_client.get_object(Bucket=bucket, Key=key)
constraints = json.load(obj['Body'])

print("Full structure:")
print(json.dumps(constraints, indent=2)[:1000])  # First 1000 chars

constraints_df = pd.json_normalize(constraints['features'])
print("\nAvailable columns:")
print(constraints_df.columns.tolist())
print("\nFirst 5 rows:")
print(constraints_df.head())

Full structure:
{
  "version": 0.0,
  "features": [
    {
      "name": "meanfreq",
      "inferred_type": "Fractional",
      "completeness": 1.0,
      "num_constraints": {
        "is_non_negative": true
      }
    },
    {
      "name": "sd",
      "inferred_type": "Fractional",
      "completeness": 1.0,
      "num_constraints": {
        "is_non_negative": true
      }
    },
    {
      "name": "median",
      "inferred_type": "Fractional",
      "completeness": 1.0,
      "num_constraints": {
        "is_non_negative": true
      }
    },
    {
      "name": "q25",
      "inferred_type": "Fractional",
      "completeness": 1.0,
      "num_constraints": {
        "is_non_negative": true
      }
    },
    {
      "name": "q75",
      "inferred_type": "Fractional",
      "completeness": 1.0,
      "num_constraints": {
        "is_non_negative": false
      }
    },
    {
      "name": "iqr",
      "inferred_type": "Fractional",
      "completeness": 1.0,
      "num_constraints":

In [11]:
#Check what statistics look like
key = "models/benchmarks/monitoring/baseline/statistics.json"
obj = s3_client.get_object(Bucket=bucket, Key=key)
statistics = json.load(obj['Body'])

stats_df = pd.json_normalize(statistics['features'])
print("Audio Feature Statistics (Top 10):")
print(stats_df[['name', 
                'numerical_statistics.mean', 
                'numerical_statistics.std_dev', 
                'numerical_statistics.min', 
                'numerical_statistics.max',
                'numerical_statistics.completeness']].head(10))

Audio Feature Statistics (Top 10):
       name  numerical_statistics.mean  numerical_statistics.std_dev  \
0  meanfreq                2348.249412                   1468.419325   
1        sd                2447.622411                   1105.571653   
2    median                4593.042497                   2696.537284   
3       q25                   0.090211                      0.045230   
4       q75                -404.312476                    106.927208   
5       iqr                  87.612007                     25.563514   
6      skew                  19.789872                     22.195497   
7      kurt                  17.906020                     11.754694   
8    sp_ent                   3.565427                     10.358779   
9       sfm                   1.530392                      7.320638   

   numerical_statistics.min  numerical_statistics.max  \
0                  0.000000               7655.335725   
1                  0.000000               6324.351767   
2

## Data Quality Monitoring

In [12]:
#Ensure monitor isn't already in place
try:
    dq_monitor.delete_monitoring_schedule(schedule_name_xgb_dq)
    print(f"Deleted existing: {schedule_name_xgb_dq}")
except:
    print("No existing schedule")

No existing schedule


### Set Up Data Quality Monitor & Schedule

In [13]:
role = get_execution_role()

#Create the monitor
dq_monitor = DefaultModelMonitor(
    role=role,
    instance_count=1,
    instance_type='ml.m5.xlarge',
    volume_size_in_gb=20,
    max_runtime_in_seconds=3600,
)

#Create schedule (using daily instead of hourly to save costs!)
schedule_name_xgb_dq = "xgb-data-quality-schedule"

#Hourly Monitoring Schedule
dq_monitor.create_monitoring_schedule(
    monitor_schedule_name=schedule_name_xgb_dq,
    endpoint_input=new_xgb_endpoint_name,  # Make sure this variable is defined
    output_s3_uri=f"s3://{bucket}/{prefix}/monitoring/data-quality/xgb",
    statistics="s3://sagemaker-us-east-1-418418308994/models/benchmarks/monitoring/baseline/statistics.json",
    constraints="s3://sagemaker-us-east-1-418418308994/models/benchmarks/monitoring/baseline/constraints.json",
    schedule_cron_expression=CronExpressionGenerator.hourly(), #Hourly
    enable_cloudwatch_metrics=True,
)

print(f"Daily schedule live: {schedule_name_xgb_dq}")

Daily schedule live: xgb-data-quality-schedule


In [14]:
#Check status of endpoint
sm_client = boto3.client('sagemaker')
response = sm_client.describe_endpoint(EndpointName=new_xgb_endpoint_name)
print("Status:", response['EndpointStatus'])
print("Last heartbeat:", response.get('LastHeartbeatTimestamp', 'N/A'))

Status: InService
Last heartbeat: N/A


## Model Quality Moniter

In [16]:
#Get predictions (batch all rows)
X_baseline = baseline_df.drop('target', axis=1)
csv_payload = X_baseline.to_csv(header=False, index=False)

#Get predictions from endpoint
response = xg_predictor_new.predict(  
    data=csv_payload,
    initial_args={'ContentType': 'text/csv', 'Accept': 'text/csv'}
)

# Parse CSV predictions (format: prob1,prob2,...,prob7\n per row)
predictions_text = response.decode('utf-8').strip().split('\n')
prob_matrix = np.array([[float(x) for x in line.split(',') if x] for line in predictions_text])
pred_labels = np.argmax(prob_matrix, axis=1)

# Create MQ baseline dataframe
mq_baseline_df = pd.DataFrame({
    'prediction': pred_labels,
    'ground_truth_label': baseline_df['target'].values
})

# Save using s3_client instead
mq_baseline_key = "models/benchmarks/mq_baseline.csv"
csv_buffer = StringIO()
mq_baseline_df.to_csv(csv_buffer, index=False, header=True)
s3_client.put_object(
    Bucket=bucket,
    Key=mq_baseline_key,
    Body=csv_buffer.getvalue()
)
mq_baseline_uri = f"s3://{bucket}/{mq_baseline_key}"
print(f"MQ baseline: {mq_baseline_uri} ({len(mq_baseline_df)} rows)")

MQ baseline: s3://sagemaker-us-east-1-418418308994/models/benchmarks/mq_baseline.csv (10242 rows)


In [39]:
# Send predictions (all at once for speed)
csv_payload = test_data.to_csv(header=False, index=False)

try:
    response = xg_predictor_new.predict(
        data=csv_payload,
        initial_args={'ContentType': 'text/csv', 'Accept': 'text/csv'}
    )
    print(f"✅ Sent {len(test_data)} predictions successfully!")
    
    # Show sample output
    print(f"\nSample predictions (first 3):")
    predictions = response.split('\n')[:3]
    for i, pred in enumerate(predictions, 1):
        if pred.strip():
            probs = [float(x) for x in pred.split(',')]
            predicted_class = probs.index(max(probs))
            print(f"  {i}. Predicted class: {predicted_class} (probs: {[f'{p:.3f}' for p in probs[:3]]}...)")
    
except Exception as e:
    print(f"❌ Error: {e}")


print("""
✅ Predictions sent!
""")

print(f"\nDataCapture location:")
print(f"s3://{bucket}/{prefix}/datacapture/{new_xgb_endpoint_name}/AllTraffic/")

✅ Sent 100 predictions successfully!

Sample predictions (first 3):
❌ Error: a bytes-like object is required, not 'str'

NEXT STEPS

✅ Predictions sent!

Now:
  1. ⏳ Wait 5 minutes for DataCapture to write to S3
  2. 🔍 Re-run the verification script to confirm datacapture exists
  3. ⏰ Wait for next hourly monitoring run
  4. 📊 Check results in S3 after monitoring completes


DataCapture location:
s3://sagemaker-us-east-1-418418308994/models/benchmarks/datacapture/xgb-benchmark-endpoint-20260210-045052/AllTraffic/


In [40]:
 import time

print("\n⏳ Waiting 5 minutes for DataCapture to write to S3...")
print("   (DataCapture writes in batches every few minutes)")

for i in range(5):
    time.sleep(60)  # Wait 1 minute
    print(f"   {i+1}/5 minutes elapsed...")

print("\n✅ 5 minutes passed. Checking DataCapture now...")

# Now verify datacapture exists
from datetime import datetime, timedelta, timezone

fs = s3fs.S3FileSystem()
bucket = "sagemaker-us-east-1-418418308994"
prefix = "models/benchmarks"
endpoint_name = "xgb-benchmark-endpoint-20260210-045052"  # Your new endpoint

target_hour = datetime.now(timezone.utc).replace(minute=0, second=0, microsecond=0)

print("VERIFYING DATACAPTURE")

datacapture_path = f"{bucket}/{prefix}/datacapture/{endpoint_name}/AllTraffic/{target_hour.strftime('%Y/%m/%d/%H')}/"

print(f"\n📍 Looking in: s3://{datacapture_path}")

try:
    dc_files = fs.ls(datacapture_path)
    jsonl_files = [f for f in dc_files if f.endswith('.jsonl')]
    
    if jsonl_files:
        print(f"✅ SUCCESS! Found {len(jsonl_files)} datacapture files")
        
        # Count total predictions
        total_predictions = 0
        for f in jsonl_files[:5]:  # Check first 5 files
            with fs.open(f, 'r') as file:
                lines = file.readlines()
                total_predictions += len(lines)
            print(f"   - {f.split('/')[-1]}: {len(lines)} predictions")
        
        print(f"\n   Total predictions captured: {total_predictions}")
        
        # Show sample
        with fs.open(jsonl_files[0], 'r') as f:
            sample = json.loads(f.readline())
        
        pred_time = sample.get('eventMetadata', {}).get('inferenceTime')
        print(f"   Sample timestamp: {pred_time}")
        
        print(f"\n✅ DATACAPTURE VERIFIED!")
        
    else:
        print(f"⚠️  No datacapture files yet for {target_hour.strftime('%H:00 UTC')}")
        print(f"\n💡 Possible reasons:")
        print(f"   1. Still writing (wait another 2-3 minutes)")
        print(f"   2. Predictions sent in previous hour")
        
        # Check previous hour
        prev_hour = target_hour - timedelta(hours=1)
        prev_path = f"{bucket}/{prefix}/datacapture/{endpoint_name}/AllTraffic/{prev_hour.strftime('%Y/%m/%d/%H')}/"
        
        try:
            prev_files = fs.ls(prev_path)
            prev_jsonl = [f for f in prev_files if f.endswith('.jsonl')]
            
            if prev_jsonl:
                print(f"\n✅ Found {len(prev_jsonl)} files in PREVIOUS hour ({prev_hour.strftime('%H:00 UTC')})")
                print(f"   Your predictions were captured there!")
        except:
            pass
        
except FileNotFoundError:
    print(f"❌ Directory doesn't exist yet")
    print(f"\n⏳ DataCapture is still writing. Wait 2-3 more minutes.")
    
except Exception as e:
    print(f"❌ Error: {e}")

# Check both current and previous hour ground truth
print("\n" + "="*70)
print("CHECKING GROUND TRUTH ALIGNMENT")
print("="*70)

for hour_offset in [0, 1]:
    check_hour = target_hour - timedelta(hours=hour_offset)
    gt_path = f"{bucket}/{prefix}/ground-truth/{check_hour.strftime('%Y/%m/%d/%H')}/"
    
    print(f"\n📅 {check_hour.strftime('%Y-%m-%d %H:00 UTC')}")
    
    try:
        gt_files = fs.ls(gt_path)
        dc_path = f"{bucket}/{prefix}/datacapture/{endpoint_name}/AllTraffic/{check_hour.strftime('%Y/%m/%d/%H')}/"
        dc_files = fs.ls(dc_path)
        dc_jsonl = [f for f in dc_files if f.endswith('.jsonl')]
        
        gt_status = "✅" if gt_files else "❌"
        dc_status = "✅" if dc_jsonl else "❌"
        
        print(f"   Ground Truth: {gt_status}")
        print(f"   DataCapture:  {dc_status}")
        
        if gt_files and dc_jsonl:
            print(f"   🎉 READY FOR MONITORING!")
            
    except:
        print(f"   ⚠️  Missing data")

print("FINAL STATUS")
print(f"""
✅ Predictions sent
⏳ Checking DataCapture status...

Next:
  1. If DataCapture confirmed above → Wait for hourly monitoring run
  2. If not yet confirmed → Wait 2-3 more minutes and re-check
  3. Monitoring runs at: {(target_hour + timedelta(hours=1)).strftime('%H:00 UTC')}
  4. Results appear: ~{(target_hour + timedelta(hours=1, minutes=10)).strftime('%H:%M UTC')}
""")

WAITING FOR DATACAPTURE

⏳ Waiting 5 minutes for DataCapture to write to S3...
   (DataCapture writes in batches every few minutes)
   1/5 minutes elapsed...
   2/5 minutes elapsed...
   3/5 minutes elapsed...
   4/5 minutes elapsed...
   5/5 minutes elapsed...

✅ 5 minutes passed. Checking DataCapture now...

VERIFYING DATACAPTURE

📍 Looking in: s3://sagemaker-us-east-1-418418308994/models/benchmarks/datacapture/xgb-benchmark-endpoint-20260210-045052/AllTraffic/2026/02/10/05/
❌ Directory doesn't exist yet

⏳ DataCapture is still writing. Wait 2-3 more minutes.

CHECKING GROUND TRUTH ALIGNMENT

📅 2026-02-10 05:00 UTC
   ⚠️  Missing data

📅 2026-02-10 04:00 UTC
   ⚠️  Missing data

FINAL STATUS

✅ Predictions sent
⏳ Checking DataCapture status...

Next:
 1. If DataCapture confirmed above → Wait for hourly monitoring run
 2. If not yet confirmed → Wait 2-3 more minutes and re-check
 3. Monitoring runs at: 06:00 UTC
 4. Results appear: ~06:10 UTC



In [17]:
# Create Model Quality Monitor
mq_monitor = ModelQualityMonitor(
    role=role, 
    instance_count=1, 
    instance_type="ml.m5.xlarge",
    volume_size_in_gb=20, 
    max_runtime_in_seconds=1800, 
    sagemaker_session=session,
)

# Define problem type and constraints
model_quality_baseline_uri = f"s3://{bucket}/{prefix}/monitoring/mq-baseline"

In [18]:
#Suggest baseline for model quality
mq_monitor.suggest_baseline(
    baseline_dataset=mq_baseline_uri,
    dataset_format=DatasetFormat.csv(header=True),
    output_s3_uri=model_quality_baseline_uri,
    problem_type='MulticlassClassification',
    inference_attribute='prediction',  
    ground_truth_attribute='ground_truth_label',  
    wait=True,
    logs=False
)

print("\n✅✅ \nModel Quality baseline created! \n✅✅")

INFO:sagemaker:Creating processing-job with name baseline-suggestion-job-2026-02-10-04-59-29-654


...........................................................!
✅✅ 
Model Quality baseline created! 
✅✅


In [19]:
# Get the baselining job description
job_description = mq_monitor.latest_baselining_job.describe()

# Extract the S3 URIs
mq_stats_uri = job_description['ProcessingOutputConfig']['Outputs'][0]['S3Output']['S3Uri'] + '/statistics.json'
mq_constraints_uri = job_description['ProcessingOutputConfig']['Outputs'][0]['S3Output']['S3Uri'] + '/constraints.json'

print(f"Stats: {mq_stats_uri}")
print(f"Constraints: {mq_constraints_uri}")

Stats: s3://sagemaker-us-east-1-418418308994/models/benchmarks/monitoring/mq-baseline/statistics.json
Constraints: s3://sagemaker-us-east-1-418418308994/models/benchmarks/monitoring/mq-baseline/constraints.json


In [42]:
s3_resource = boto3.resource('s3')

# Get the hour to upload for (current hour)
target_hour = datetime.utcnow().replace(minute=0, second=0, microsecond=0)

# CORRECT PATH: Use hierarchical structure with year/month/day/hour folders
# The file itself can be named anything
ground_truth_s3_key = f"{prefix}/ground-truth/{target_hour.strftime('%Y/%m/%d/%H')}/ground-truth.jsonl"

print(f"📅 Uploading ground truth for: {target_hour.strftime('%Y-%m-%d %H:00 UTC')}")
print(f"📍 S3 Path: s3://{bucket}/{ground_truth_s3_key}")

# Generate ground truth records
fake_gt = []
for i, gt_label in enumerate(baseline_df['target'].iloc[:100]):
    fake_gt.append(json.dumps({
        "groundTruthData": {
            "data": str(int(gt_label)),
            "encoding": "CSV"
        },
        "eventMetadata": {
            "eventId": f"lab-{i}",
            "inferenceTime": target_hour.isoformat()
        },
        "eventVersion": "0"
    }))

# Upload to S3 with correct hierarchical path
s3_resource.Object(bucket, ground_truth_s3_key).put(Body='\n'.join(fake_gt))

print(f"✅ Uploaded {len(fake_gt)} ground truth records")
print(f"\n💡 Monitoring will look for files in:")
print(f"   s3://{bucket}/{prefix}/ground-truth/{target_hour.strftime('%Y/%m/%d/%H')}/")  # ← Fixed!
print(f"   Found: ground-truth.jsonl ✅")

📅 Uploading ground truth for: 2026-02-10 05:00 UTC
📍 S3 Path: s3://sagemaker-us-east-1-418418308994/models/benchmarks/ground-truth/2026/02/10/05/ground-truth.jsonl
✅ Uploaded 100 ground truth records

💡 Monitoring will look for files in:
   s3://sagemaker-us-east-1-418418308994/models/benchmarks/ground-truth/2026/02/09/07/
   Found: ground-truth.jsonl ✅


/tmp/ipykernel_395/3016814757.py:4: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  target_hour = datetime.utcnow().replace(minute=0, second=0, microsecond=0)


In [45]:
import s3fs
from datetime import datetime, timezone
import pandas as pd

fs = s3fs.S3FileSystem()
bucket = "sagemaker-us-east-1-418418308994"
prefix = "models/benchmarks"
endpoint_name = new_xgb_endpoint_name

# CURRENT hour (not hardcoded!)
now = datetime.now(timezone.utc)
current_hour = now.replace(minute=0, second=0, microsecond=0)

print("="*70)
print(f"VERIFICATION - {now.strftime('%H:%M UTC')}")
print("="*70)

print(f"\n📅 Checking hour: {current_hour.strftime('%Y-%m-%d %H:00 UTC')}")

# 1. Check DataCapture
dc_path = f"{bucket}/{prefix}/datacapture/{endpoint_name}/AllTraffic/{current_hour.strftime('%Y/%m/%d/%H')}/"

print(f"\n1️⃣ DataCapture:")
try:
    dc_files = [f for f in fs.ls(dc_path) if f.endswith('.jsonl')]
    print(f"   {'✅' if dc_files else '❌'} {len(dc_files)} files")
    if not dc_files:
        print(f"   Action: Send predictions NOW")
except:
    print(f"   ❌ Not found - send predictions!")

# 2. Check Ground Truth
gt_path = f"{bucket}/{prefix}/ground-truth/{current_hour.strftime('%Y/%m/%d/%H')}/"

print(f"\n2️⃣ Ground Truth:")
try:
    gt_files = fs.ls(gt_path)
    print(f"   {'✅' if gt_files else '❌'} {len(gt_files)} files")
    if not gt_files:
        print(f"   Action: Upload ground truth for {current_hour.strftime('%H:00')}")
except:
    print(f"   ❌ Not found - upload ground truth!")

# 3. Summary
print(f"\n{'='*70}")

has_dc = len(dc_files) if 'dc_files' in locals() and dc_files else 0
has_gt = len(gt_files) if 'gt_files' in locals() and gt_files else 0

if has_dc and has_gt:
    print("🎉 READY! Monitoring will run next hour.")
elif not has_dc:
    print("⚠️  SEND PREDICTIONS NOW")
    print("""
# Run this:
predictor = Predictor(endpoint_name=new_xgb_endpoint_name, sagemaker_session=session)
baseline_df = pd.read_csv(f's3://{bucket}/{prefix}/baseline_normalized.csv')
test_data = baseline_df.drop('target', axis=1).iloc[:100]
csv_payload = test_data.to_csv(header=False, index=False)
response = predictor.predict(data=csv_payload, initial_args={'ContentType': 'text/csv'})
print('✅ Predictions sent! Wait 5 minutes.')
    """)
elif not has_gt:
    print("⚠️  UPLOAD GROUND TRUTH NOW")
    print(f"   (Already showed you this code above)")

VERIFICATION - 05:58 UTC

📅 Checking hour: 2026-02-10 05:00 UTC

1️⃣ DataCapture:
   ❌ Not found - send predictions!

2️⃣ Ground Truth:
   ✅ 1 files

⚠️  SEND PREDICTIONS NOW

# Run this:
predictor = Predictor(endpoint_name=new_xgb_endpoint_name, sagemaker_session=session)
baseline_df = pd.read_csv(f's3://{bucket}/{prefix}/baseline_normalized.csv')
test_data = baseline_df.drop('target', axis=1).iloc[:100]
csv_payload = test_data.to_csv(header=False, index=False)
response = predictor.predict(data=csv_payload, initial_args={'ContentType': 'text/csv'})
print('✅ Predictions sent! Wait 5 minutes.')
    


In [46]:
from sagemaker.predictor import Predictor
import pandas as pd

print("SENDING PREDICTIONS FOR HOUR 05:00")

# Create predictor
predictor = Predictor(
    endpoint_name=new_xgb_endpoint_name, 
    sagemaker_session=session
)

# Load data
baseline_df = pd.read_csv(f's3://{bucket}/{prefix}/baseline_normalized.csv')
test_data = baseline_df.drop('target', axis=1).iloc[:100]

# Send predictions
csv_payload = test_data.to_csv(header=False, index=False)

print(f"\n📤 Sending 100 predictions to {new_xgb_endpoint_name}...")

response = predictor.predict(
    data=csv_payload, 
    initial_args={'ContentType': 'text/csv', 'Accept': 'text/csv'}
)

print('✅ Predictions sent!')
print('\n⏳ Wait 5 minutes for DataCapture to write...')

# Wait and verify
import time
for i in range(5):
    time.sleep(60)
    print(f'   {i+1}/5 minutes...')

print('\n🔍 Verifying DataCapture...')

import s3fs
from datetime import datetime, timezone

fs = s3fs.S3FileSystem()
current_hour = datetime.now(timezone.utc).replace(minute=0, second=0, microsecond=0)
dc_path = f"{bucket}/{prefix}/datacapture/{new_xgb_endpoint_name}/AllTraffic/{current_hour.strftime('%Y/%m/%d/%H')}/"

try:
    dc_files = [f for f in fs.ls(dc_path) if f.endswith('.jsonl')]
    if dc_files:
        print(f'✅ SUCCESS! Found {len(dc_files)} datacapture files')
        print(f'\n🎉 READY FOR MONITORING!')
        print(f'⏰ Monitoring runs at: 06:00 UTC')
        print(f'📊 Results at: ~06:10 UTC')
    else:
        print('⚠️  Files not written yet - wait another 2-3 minutes')
except Exception as e:
    print(f'⚠️  Not found yet: {e}')
    print('   Wait another 2-3 minutes and check manually')

SENDING PREDICTIONS FOR HOUR 05:00

📤 Sending 100 predictions to xgb-benchmark-endpoint-20260210-045052...
✅ Predictions sent!

⏳ Wait 5 minutes for DataCapture to write...
   1/5 minutes...
   2/5 minutes...
   3/5 minutes...
   4/5 minutes...
   5/5 minutes...

🔍 Verifying DataCapture...
⚠️  Not found yet: sagemaker-us-east-1-418418308994/models/benchmarks/datacapture/xgb-benchmark-endpoint-20260210-045052/AllTraffic/2026/02/10/06
   Wait another 2-3 minutes and check manually


In [22]:
# Create monitor schedule
mq_monitor.create_monitoring_schedule(
    monitor_schedule_name="xgb-model-quality-schedule",
    endpoint_input=EndpointInput(
        endpoint_name=new_xgb_endpoint_name,
        destination="/opt/ml/processing/input/endpoint",
        inference_attribute="0"
    ),
    problem_type="MulticlassClassification",
    ground_truth_input=f"s3://{bucket}/{prefix}/ground-truth",
    output_s3_uri=f"s3://{bucket}/{prefix}/monitoring/model-quality",
    schedule_cron_expression=CronExpressionGenerator.hourly(),
    enable_cloudwatch_metrics=True
)

print("✅ Model Quality monitoring schedule created")

INFO:sagemaker.model_monitor.model_monitoring:Creating Monitoring Schedule with name: xgb-model-quality-schedule


✅ Model Quality monitoring schedule created


## Model Bias Monitor

In [23]:
print("Skipping bias monitoring - no demographic features in dataset")

Skipping bias monitoring - no demographic features in dataset


## Infrastructure Monitors

In [24]:
# Create CloudWatch client
cw_client = boto3.client('cloudwatch')

# Infrastructure monitoring - CloudWatch alarms for XGBoost endpoint
endpoint_metric_dimensions = [
    {"Name": "EndpointName", "Value": new_xgb_endpoint_name},  # Using your current endpoint variable
]

# Alarm 1: High latency
cw_client.put_metric_alarm(
    AlarmName="XGB-Endpoint-High-Latency",
    AlarmDescription="Model latency for XGBoost endpoint above threshold",
    Namespace="AWS/SageMaker",
    MetricName="ModelLatency",
    Dimensions=endpoint_metric_dimensions,
    Statistic="Average",
    Period=300,  # 5 minutes
    EvaluationPeriods=2,
    Threshold=10000.0,  # 10 seconds in milliseconds
    ComparisonOperator="GreaterThanThreshold",
    TreatMissingData="notBreaching",
    ActionsEnabled=False,  # Set to True if you want to enable SNS notifications
)
print("CloudWatch alarm created for XGBoost endpoint latency")

# Alarm 2: Invocation errors
cw_client.put_metric_alarm(
    AlarmName="XGB-Endpoint-Invocation-Errors",
    AlarmDescription="Invocation errors for XGBoost endpoint",
    Namespace="AWS/SageMaker",
    MetricName="ModelInvocationErrors",
    Dimensions=endpoint_metric_dimensions,
    Statistic="Sum",
    Period=300,  # 5 minutes
    EvaluationPeriods=1,
    Threshold=5.0,  # Alert if more than 5 errors in 5 minutes
    ComparisonOperator="GreaterThanThreshold",
    TreatMissingData="notBreaching",
    ActionsEnabled=False,  # Set to True if you want to enable SNS notifications
)
print("CloudWatch alarm created for XGBoost endpoint invocation errors")

# Alarm 3: 5XX server errors
cw_client.put_metric_alarm(
    AlarmName="XGB-Endpoint-5XX-Errors-High",
    AlarmDescription="5XX error rate for XGBoost endpoint above threshold",
    Namespace="AWS/SageMaker",
    MetricName="Model5XXErrors",  # Changed from ModelLatency
    Dimensions=endpoint_metric_dimensions,
    Statistic="Sum",
    Period=300,  # 5 minutes
    EvaluationPeriods=2,
    Threshold=5.0,  # Alert if more than 5 5XX errors
    ComparisonOperator="GreaterThanThreshold",
    TreatMissingData="notBreaching",
    ActionsEnabled=False,
)
print("CloudWatch alarm created for XGBoost endpoint 5XX errors")

# Alarm 4: 4XX client errors
cw_client.put_metric_alarm(
    AlarmName="XGB-Endpoint-4XX-Errors-High",
    AlarmDescription="4XX error rate for XGBoost endpoint above threshold",
    Namespace="AWS/SageMaker",
    MetricName="Model4XXErrors",
    Dimensions=endpoint_metric_dimensions,
    Statistic="Sum",
    Period=300,  # 5 minutes
    EvaluationPeriods=2,
    Threshold=10.0,  # Alert if more than 10 4XX errors
    ComparisonOperator="GreaterThanThreshold",
    TreatMissingData="notBreaching",
    ActionsEnabled=False,
)
print("CloudWatch alarm created for XGBoost endpoint 4XX errors")

CloudWatch alarm created for XGBoost endpoint latency
CloudWatch alarm created for XGBoost endpoint invocation errors
CloudWatch alarm created for XGBoost endpoint 5XX errors
CloudWatch alarm created for XGBoost endpoint 4XX errors


In [25]:
#Set each alarm state manually as a test
cw_client = boto3.client('cloudwatch')

# Test each alarm by manually setting it to ALARM state
alarms_to_test = [
    "XGB-Endpoint-High-Latency",
    "XGB-Endpoint-Invocation-Errors",
    "XGB-Endpoint-5XX-Errors-High",
    "XGB-Endpoint-4XX-Errors-High"
]

print("Testing alarms by triggering them manually...\n")

for alarm_name in alarms_to_test:
    # Set alarm to ALARM state
    cw_client.set_alarm_state(
        AlarmName=alarm_name,
        StateValue='ALARM',
        StateReason='Testing alarm - manually triggered from notebook',
        StateReasonData=f'{{"test": "true", "timestamp": "{datetime.now().isoformat()}"}}'
    )
    print(f"Triggered: {alarm_name} -> ALARM state")
    
    # Check the alarm state
    response = cw_client.describe_alarms(AlarmNames=[alarm_name])
    alarm = response['MetricAlarms'][0]
    print(f"   State: {alarm['StateValue']}")
    print(f"   Reason: {alarm['StateReason']}\n")

print("\n" + "="*60)
print("All alarms triggered! Check CloudWatch console to verify.")
print("="*60)

# Optional: Reset all alarms back to OK after testing
print("\nResetting alarms to OK state...\n")
for alarm_name in alarms_to_test:
    cw_client.set_alarm_state(
        AlarmName=alarm_name,
        StateValue='OK',
        StateReason='Test completed - resetting alarm',
    )
    print(f"Reset: {alarm_name} -> OK state")

Testing alarms by triggering them manually...

Triggered: XGB-Endpoint-High-Latency -> ALARM state
   State: ALARM
   Reason: Testing alarm - manually triggered from notebook

Triggered: XGB-Endpoint-Invocation-Errors -> ALARM state
   State: ALARM
   Reason: Testing alarm - manually triggered from notebook

Triggered: XGB-Endpoint-5XX-Errors-High -> ALARM state
   State: ALARM
   Reason: Testing alarm - manually triggered from notebook

Triggered: XGB-Endpoint-4XX-Errors-High -> ALARM state
   State: ALARM
   Reason: Testing alarm - manually triggered from notebook


All alarms triggered! Check CloudWatch console to verify.

Resetting alarms to OK state...

Reset: XGB-Endpoint-High-Latency -> OK state
Reset: XGB-Endpoint-Invocation-Errors -> OK state
Reset: XGB-Endpoint-5XX-Errors-High -> OK state
Reset: XGB-Endpoint-4XX-Errors-High -> OK state


## CloudWatch Monitoring Dashboard

In [26]:
cw_client = boto3.client('cloudwatch')

dashboard_name = "SageMaker-ML-Benchmarks"
dashboard_body = {
    "widgets": [
        {
            "type": "metric",
            "x": 0,
            "y": 0,
            "width": 12,
            "height": 6,
            "properties": {
                "title": "XGBoost Endpoint – Invocations & Latency",
                "metrics": [
                    ["AWS/SageMaker", "Invocations", "EndpointName", new_xgb_endpoint_name],
                    [".", "ModelLatency", ".", "."],
                ],
                "stacked": False,
                "stat": "Average",
                "period": 60,
                "region": region,
            },
        },
        {
            "type": "metric",
            "x": 12,
            "y": 0,
            "width": 12,
            "height": 6,
            "properties": {
                "title": "XGBoost Endpoint – Errors",
                "metrics": [
                    ["AWS/SageMaker", "ModelInvocationErrors", "EndpointName", new_xgb_endpoint_name],
                    [".", "Invocation4XXErrors", ".", "."],
                    [".", "Invocation5XXErrors", ".", "."],
                ],
                "stacked": False,
                "stat": "Sum",
                "period": 300,
                "region": region,
            },
        },
        {
            "type": "metric",
            "x": 0,
            "y": 6,
            "width": 12,
            "height": 6,
            "properties": {
                "title": "XGBoost Data Quality Violations",
                "metrics": [
                    [
                        "AWS/SageMaker",
                        "DataQualityViolation",
                        "MonitoringSchedule",
                        "xgb-data-quality-schedule",
                    ],
                ],
                "stat": "Sum",
                "period": 300,
                "region": region,
            },
        },
        {
            "type": "metric",
            "x": 12,
            "y": 6,
            "width": 12,
            "height": 6,
            "properties": {
                "title": "XGBoost Model Quality Violations",
                "metrics": [
                    [
                        "AWS/SageMaker",
                        "ModelQualityViolation",
                        "MonitoringSchedule",
                        "xgb-model-quality-schedule",
                    ],
                ],
                "stat": "Sum",
                "period": 300,
                "region": region,
            },
        },
        {
            "type": "metric",
            "x": 0,
            "y": 12,
            "width": 24,
            "height": 6,
            "properties": {
                "title": "CloudWatch Alarms Status",
                "metrics": [
                    ["AWS/CloudWatch", "AlarmState", "AlarmName", "XGB-Endpoint-High-Latency"],
                    ["...", "XGB-Endpoint-Invocation-Errors"],
                    ["...", "XGB-Endpoint-5XX-Errors-High"],
                    ["...", "XGB-Endpoint-4XX-Errors-High"],
                ],
                "stat": "Maximum",
                "period": 300,
                "region": region,
            },
        },
    ]
}

cw_client.put_dashboard(
    DashboardName=dashboard_name,
    DashboardBody=json.dumps(dashboard_body),
)

print(f"CloudWatch Dashboard created: {dashboard_name}")
print(f"View at: https://console.aws.amazon.com/cloudwatch/home?region={region}#dashboards:name={dashboard_name}")

CloudWatch Dashboard created: SageMaker-ML-Benchmarks
View at: https://console.aws.amazon.com/cloudwatch/home?region=us-east-1#dashboards:name=SageMaker-ML-Benchmarks


## Generate Model Traffic

In [28]:
print("Generating data quality violations...")

# Load your baseline to see normal ranges
baseline_df = pd.read_csv("s3://sagemaker-us-east-1-418418308994/models/benchmarks/baseline_normalized.csv")
X_baseline = baseline_df.drop('target', axis=1)

# Create anomalous data - extreme values outside normal ranges
anomalous_data = X_baseline.copy()[:10]

# Introduce violations by setting extreme values
for col in anomalous_data.columns[:5]:  # Corrupt first 5 features
    anomalous_data[col] = anomalous_data[col].max() * 100  # 100x the max value
    
print(f"Anomalous data shape: {anomalous_data.shape}")
print(f"Sample extreme values:\n{anomalous_data.iloc[0, :5]}")

# Send anomalous data to endpoint
csv_payload = anomalous_data.to_csv(header=False, index=False)

for i in range(5):
    response = xg_predictor_new.predict(
        data=csv_payload,
        initial_args={'ContentType': 'text/csv', 'Accept': 'text/csv'}
    )
    print(f"Sent anomalous request {i+1}/5")
    time.sleep(1)

print("\n✅ Anomalous data sent! This should trigger data quality violations on next monitoring run.")

Generating data quality violations...
Anomalous data shape: (10, 20)
Sample extreme values:
meanfreq    450709.170872
sd          419013.377644
median      854181.780134
q25             15.419224
q75         -23044.186000
Name: 0, dtype: float64
Sent anomalous request 1/5
Sent anomalous request 2/5
Sent anomalous request 3/5
Sent anomalous request 4/5
Sent anomalous request 5/5

✅ Anomalous data sent! This should trigger data quality violations on next monitoring run.


In [29]:
print("Generating model quality violations...")

# Create fake ground truth with WRONG labels to simulate poor accuracy
s3_resource = boto3.resource('s3')

# Get some predictions
X_test = baseline_df.drop('target', axis=1).iloc[:100]
csv_payload = X_test.to_csv(header=False, index=False)

response = xg_predictor_new.predict(
    data=csv_payload,
    initial_args={'ContentType': 'text/csv', 'Accept': 'text/csv'}
)

# Parse predictions
predictions_text = response.decode('utf-8').strip().split('\n')
prob_matrix = np.array([[float(x) for x in line.split(',') if x] for line in predictions_text])
pred_labels = np.argmax(prob_matrix, axis=1)

# Create INCORRECT ground truth (flip predictions to opposite class)
incorrect_ground_truth = []
for i, pred in enumerate(pred_labels):
    # Assign wrong label - pick a different class than predicted
    wrong_label = (pred + 3) % 7  # Shift by 3 classes to ensure it's wrong
    
    incorrect_ground_truth.append(json.dumps({
        "groundTruthData": {
            "data": str(int(wrong_label)),
            "encoding": "CSV"
        },
        "eventMetadata": {
            "eventId": f"violation-test-{i}",
            "inferenceTime": (datetime.now() - timedelta(hours=1)).isoformat()
        },
        "eventVersion": "0"
    }))

# Upload incorrect ground truth
ground_truth_key = f"ground-truth-violations-{datetime.now().strftime('%Y%m%d-%H%M')}.jsonl"
s3_resource.Object(bucket, f"{prefix}/ground-truth/{ground_truth_key}").put(
    Body='\n'.join(incorrect_ground_truth)
)

print(f"✅\n✅\n✅\n Uploaded incorrect ground truth: s3://{bucket}/{prefix}/ground-truth/{ground_truth_key}")
print("This should trigger model quality violations on next monitoring run (shows 0% accuracy).")

Generating model quality violations...
✅
✅
✅
 Uploaded incorrect ground truth: s3://sagemaker-us-east-1-418418308994/models/benchmarks/ground-truth/ground-truth-violations-20260210-0509.jsonl
This should trigger model quality violations on next monitoring run (shows 0% accuracy).


In [30]:
# Manually trigger alarms to show in dashboard
alarms_to_test = [
    "XGB-Endpoint-High-Latency",
    "XGB-Endpoint-Invocation-Errors",
    "XGB-Endpoint-5XX-Errors-High",
    "XGB-Endpoint-4XX-Errors-High"
]

for alarm_name in alarms_to_test:
    cw_client.set_alarm_state(
        AlarmName=alarm_name,
        StateValue='ALARM',
        StateReason='Dashboard test - manually triggered',
    )
    print(f"Triggered: {alarm_name}")

print("\nRefresh dashboard to see alarm states!")

Triggered: XGB-Endpoint-High-Latency
Triggered: XGB-Endpoint-Invocation-Errors
Triggered: XGB-Endpoint-5XX-Errors-High
Triggered: XGB-Endpoint-4XX-Errors-High

Refresh dashboard to see alarm states!


In [48]:
#Send data to Dashboard
# 1. Send batch predictions (for model quality)
print("\n1️⃣ Sending batch predictions...")
csv_payload = test_data.to_csv(header=False, index=False)
response = predictor.predict(
    data=csv_payload,
    initial_args={'ContentType': 'text/csv', 'Accept': 'text/csv'}
)
print("✅ Sent 100 predictions (1 invocation)")

# 2. Send individual requests (for dashboard metrics)
print("\n2️⃣ Sending individual requests for dashboard...")
for i in range(20):
    try:
        single_payload = test_data.iloc[i:i+1].to_csv(header=False, index=False)
        response = predictor.predict(
            data=single_payload,
            initial_args={'ContentType': 'text/csv', 'Accept': 'text/csv'}
        )
        print(f"   Request {i+1}/20 completed")
        time.sleep(2)  # Spread out over time
    except Exception as e:
        print(f"   Request {i+1} failed: {e}")

print("\n✅ Traffic generated!")
print("\n⏳ Wait 2-3 minutes, then refresh your dashboard")
print("   CloudWatch metrics take a few minutes to appear")


1️⃣ Sending batch predictions...
✅ Sent 100 predictions (1 invocation)

2️⃣ Sending individual requests for dashboard...
   Request 1/20 completed
   Request 2/20 completed
   Request 3/20 completed
   Request 4/20 completed
   Request 5/20 completed
   Request 6/20 completed
   Request 7/20 completed
   Request 8/20 completed
   Request 9/20 completed
   Request 10/20 completed
   Request 11/20 completed
   Request 12/20 completed
   Request 13/20 completed
   Request 14/20 completed
   Request 15/20 completed
   Request 16/20 completed
   Request 17/20 completed
   Request 18/20 completed
   Request 19/20 completed
   Request 20/20 completed

✅ Traffic generated!

⏳ Wait 2-3 minutes, then refresh your dashboard
   CloudWatch metrics take a few minutes to appear


## Troubleshooting Quality Monitor

In [ ]:
#What does data capture look like? 
bucket = "sagemaker-us-east-1-418418308994"
prefix = "models/benchmarks"
endpoint_name = "xgb-benchmark-endpoint-20260209-023949"  # UPDATE THIS

fs = s3fs.S3FileSystem()

print(f"\nLooking for datacapture files...")
datacapture_prefix = f"{bucket}/{prefix}/datacapture/{endpoint_name}/AllTraffic"

try:
    # Find recent datacapture files
    all_files = fs.ls(datacapture_prefix, recursive=True)
    jsonl_files = [f for f in all_files if f.endswith('.jsonl')]
    
    if jsonl_files:
        print(f"✅ Found {len(jsonl_files)} datacapture files")
        
        # Show latest file
        latest_file = jsonl_files[-1]
        print(f"\n📄 Latest DataCapture File:")
        print(f"   {latest_file}")
        
        # Read and parse
        with fs.open(latest_file, 'r') as f:
            lines = f.readlines()[:5]  # First 5 records
        
        print(f"\n📊 Sample DataCapture Records (first 5):")
        print("-"*70)
        
        for i, line in enumerate(lines, 1):
            record = json.loads(line)
            
            # Extract key info
            event_id = record.get('eventMetadata', {}).get('eventId', 'N/A')
            inference_time = record.get('eventMetadata', {}).get('inferenceTime', 'N/A')
            
            # Get input/output
            capture_data = record.get('captureData', {})
            
            endpoint_input = capture_data.get('endpointInput', {})
            input_data = endpoint_input.get('data', 'N/A')
            
            endpoint_output = capture_data.get('endpointOutput', {})
            output_data = endpoint_output.get('data', 'N/A')
            
            print(f"\n{i}. Event ID: {event_id}")
            print(f"   Time: {inference_time}")
            print(f"   Input (first 100 chars): {input_data[:100]}...")
            print(f"   Output: {output_data}")
            
            # Parse output to get predicted class
            if output_data != 'N/A':
                # Output is CSV with probabilities for each class
                probs = [float(x) for x in output_data.split(',') if x.strip()]
                predicted_class = probs.index(max(probs))
                print(f"   → Predicted Class: {predicted_class}")
        
    else:
        print("❌ No datacapture files found!")
        print(f"   Expected location: s3://{datacapture_prefix}")
        print("\n💡 This means:")
        print("   - DataCapture is not enabled, OR")
        print("   - No predictions have been sent to endpoint, OR")
        print("   - Wrong endpoint name")
        
except Exception as e:
    print(f"❌ Error accessing datacapture: {e}")

In [ ]:
#What does ground truth look like? 
gt_prefix = f"{bucket}/{prefix}/ground-truth"

try:
    gt_files = fs.ls(gt_prefix, recursive=True)
    jsonl_gt = [f for f in gt_files if f.endswith('.jsonl')]
    
    if jsonl_gt:
        print(f"✅ Found {len(jsonl_gt)} ground truth files")
        
        # Show latest
        latest_gt = jsonl_gt[-1]
        print(f"\n📄 Latest Ground Truth File:")
        print(f"   {latest_gt}")
        
        # Read and parse
        with fs.open(latest_gt, 'r') as f:
            lines = f.readlines()[:5]
        
        print(f"\n📊 Sample Ground Truth Records (first 5):")
        print("-"*70)
        
        for i, line in enumerate(lines, 1):
            record = json.loads(line)
            
            event_id = record.get('eventMetadata', {}).get('eventId', 'N/A')
            inference_time = record.get('eventMetadata', {}).get('inferenceTime', 'N/A')
            
            gt_data = record.get('groundTruthData', {}).get('data', 'N/A')
            
            print(f"\n{i}. Event ID: {event_id}")
            print(f"   Time: {inference_time}")
            print(f"   Ground Truth Class: {gt_data}")
        
    else:
        print("❌ No ground truth files found!")
        print(f"   Expected location: s3://{gt_prefix}")
        
except Exception as e:
    print(f"❌ Error accessing ground truth: {e}")

In [ ]:
#Compare data capture v. ground truth
print("""
Monitoring Job Process:
-----------------------

1. Read DataCapture files for the hour
   Example: datacapture/.../2026/02/09/07/*.jsonl

2. Read Ground Truth files for the hour  
   Example: ground-truth/ground-truth-2026/02/09/07.jsonl

3. Match records by TIMESTAMP
   - DataCapture has: inferenceTime: "2026-02-09T07:30:00"
   - Ground Truth has: inferenceTime: "2026-02-09T07:30:00"
   - They MUST match!

4. Extract predictions and ground truth
   - From DataCapture: predicted_class = argmax(output probabilities)
   - From Ground Truth: actual_class = groundTruthData.data

5. Calculate metrics
   accuracy = (correct predictions) / (total predictions)
   precision, recall, f1 for each class

6. Compare to baseline
   If accuracy < baseline_accuracy → VIOLATION
""")

In [50]:
#Check why monitoring is failing
sagemaker = boto3.client('sagemaker')

# Get latest execution details
try:
    executions = sagemaker.list_monitoring_executions(
        MonitoringScheduleName='xgb-model-quality-schedule',
        MaxResults=5,
        SortOrder='Descending'
    )
    
    if executions['MonitoringExecutionSummaries']:
        latest = executions['MonitoringExecutionSummaries'][0]
        
        print(f"\n📊 Latest Monitoring Execution:")
        print(f"   Scheduled: {latest['ScheduledTime']}")
        print(f"   Status: {latest['MonitoringExecutionStatus']}")
        
        if latest['MonitoringExecutionStatus'] == 'Failed':
            print(f"   ❌ Failure Reason: {latest.get('FailureReason', 'Unknown')}")
            
            # Get processing job details
            if 'ProcessingJobArn' in latest:
                job_name = latest['ProcessingJobArn'].split('/')[-1]
                job_desc = sagemaker.describe_processing_job(ProcessingJobName=job_name)
                
                print(f"\n   Processing Job Details:")
                print(f"   Job Name: {job_name}")
                print(f"   Status: {job_desc['ProcessingJobStatus']}")
                
                if 'FailureReason' in job_desc:
                    print(f"   Failure: {job_desc['FailureReason']}")
                
                # Check CloudWatch Logs
                print(f"\n   💡 Check CloudWatch Logs:")
                print(f"   Log Group: /aws/sagemaker/ProcessingJobs")
                print(f"   Log Stream: {job_name}/...")
        
        elif latest['MonitoringExecutionStatus'] == 'Completed':
            print(f"   ✅ Execution completed successfully")
            
    else:
        print("⚠️  No monitoring executions found")
        
except Exception as e:
    print(f"❌ Error: {e}")


📊 Latest Monitoring Execution:
   Scheduled: 2026-02-10 06:00:00+00:00
   Status: Failed
   ❌ Failure Reason: Job inputs had no data

   Processing Job Details:
   Job Name: groundtruth-merge-202602100600-d081f80f835cf867bcd3661c
   Status: Completed

   💡 Check CloudWatch Logs:
   Log Group: /aws/sagemaker/ProcessingJobs
   Log Stream: groundtruth-merge-202602100600-d081f80f835cf867bcd3661c/...


### Make Sure Monitors Ran - After Top of Hour 

In [49]:
sagemaker_client = boto3.client('sagemaker')

def check_status():
    print(f"\n🕐 {time.strftime('%H:%M:%S')}")
    print("-"*50)
    
    # Model Quality
    mq_schedule = sagemaker_client.describe_monitoring_schedule(
        MonitoringScheduleName='xgb-model-quality-schedule'
    )
    mq_status = mq_schedule.get('LastMonitoringExecutionSummary', {}).get('MonitoringExecutionStatus', 'Not started')
    print(f"Model Quality: {mq_status}")
    
    # Data Quality
    dq_schedule = sagemaker_client.describe_monitoring_schedule(
        MonitoringScheduleName='xgb-data-quality-schedule'
    )
    dq_status = dq_schedule.get('LastMonitoringExecutionSummary', {}).get('MonitoringExecutionStatus', 'Not started')
    print(f"Data Quality:  {dq_status}")
    
    if mq_status == 'Completed' and dq_status == 'Completed':
        print("\n✅ Both jobs complete! Check for reports.")
        return True
    return False

# Check every 2 minutes for up to 20 minutes
for i in range(10):
    if check_status():
        break
    if i < 9:  # Don't sleep after last check
        print("⏳ Waiting 2 minutes...")
        time.sleep(120)

print("\n🎉 Monitoring complete! Run your report download script now.")


🕐 06:05:47
--------------------------------------------------
Model Quality: Failed
Data Quality:  InProgress
⏳ Waiting 2 minutes...

🕐 06:07:47
--------------------------------------------------
Model Quality: Failed
Data Quality:  CompletedWithViolations
⏳ Waiting 2 minutes...

🕐 06:09:47
--------------------------------------------------
Model Quality: Pending
Data Quality:  CompletedWithViolations
⏳ Waiting 2 minutes...

🕐 06:11:47
--------------------------------------------------
Model Quality: Pending
Data Quality:  CompletedWithViolations
⏳ Waiting 2 minutes...

🕐 06:13:47
--------------------------------------------------
Model Quality: InProgress
Data Quality:  CompletedWithViolations
⏳ Waiting 2 minutes...

🕐 06:15:48
--------------------------------------------------
Model Quality: InProgress
Data Quality:  CompletedWithViolations
⏳ Waiting 2 minutes...

🕐 06:17:48
--------------------------------------------------
Model Quality: Failed
Data Quality:  CompletedWithViolatio

╭─────────────────────────────── Traceback (most recent call last) ────────────────────────────────╮
│ in <module>:32                                                                                   │
│                                                                                                  │
│   29 │   │   break                                                                               │
│   30 │   if i < 9:  # Don't sleep after last check                                               │
│   31 │   │   print("⏳ Waiting 2 minutes...")                                                    │
│ ❱ 32 │   │   time.sleep(120)                                                                     │
│   33                                                                                             │
│   34 print("\n🎉 Monitoring complete! Run your report download script now.")                     │
│   35                                                                                             │
╰──────────────────────────────────────────────────────────────────────────────────────────────────╯
KeyboardInterrupt

### Generate Model & Data Reports in SageMaker

In [51]:
fs = s3fs.S3FileSystem()

print("="*70)
print("DOWNLOADING MONITORING REPORTS")
print("="*70)

# DATA QUALITY REPORTS
print("\n📊 DATA QUALITY REPORTS")
print("="*70)

dq_prefix = f"{bucket}/{prefix}/monitoring/data-quality/xgb"

try:
    # List all files recursively
    all_dq_files = fs.ls(dq_prefix, recursive=True)
    
    print(f"Found {len(all_dq_files)} files total\n")
    
    # Find constraint violations
    violations_files = [f for f in all_dq_files if 'constraint_violations.json' in f]
    
    if violations_files:
        print(f"✅ Found {len(violations_files)} violation report(s)\n")
        
        # Download and display latest
        latest_violations = violations_files[-1]
        print(f"📄 Latest Report: {latest_violations}\n")
        
        with fs.open(latest_violations, 'r') as f:
            violations_data = json.load(f)
        
        print("CONSTRAINT VIOLATIONS:")
        print(json.dumps(violations_data, indent=2))
        
        # Save locally
        with open('data_quality_violations.json', 'w') as f:
            json.dump(violations_data, f, indent=2)
        print("\n✅ Saved to: data_quality_violations.json")
        
    else:
        print("⚠️ No constraint_violations.json found")
        print("\nAll files found:")
        for f in all_dq_files:
            print(f"  - {f}")
    
    # Find statistics
    statistics_files = [f for f in all_dq_files if 'statistics.json' in f]
    
    if statistics_files:
        latest_stats = statistics_files[-1]
        print(f"\n📊 Latest Statistics: {latest_stats}\n")
        
        with fs.open(latest_stats, 'r') as f:
            stats_data = json.load(f)
        
        print("DATA STATISTICS:")
        print(json.dumps(stats_data, indent=2)[:1000])  # First 1000 chars
        
        # Save locally
        with open('data_quality_statistics.json', 'w') as f:
            json.dump(stats_data, f, indent=2)
        print("\n✅ Saved to: data_quality_statistics.json")

except Exception as e:
    print(f"❌ Error: {e}")


# MODEL QUALITY REPORTS
print("\n" + "="*70)
print("📈 MODEL QUALITY REPORTS")
print("="*70)

mq_prefix = f"{bucket}/{prefix}/monitoring/model-quality"

try:
    # List all files recursively
    all_mq_files = fs.ls(mq_prefix, recursive=True)
    
    print(f"Found {len(all_mq_files)} files total\n")
    
    # Find constraint violations
    mq_violations_files = [f for f in all_mq_files if 'constraint_violations.json' in f]
    
    if mq_violations_files:
        print(f"✅ Found {len(mq_violations_files)} violation report(s)\n")
        
        # Download and display latest
        latest_mq_violations = mq_violations_files[-1]
        print(f"📄 Latest Report: {latest_mq_violations}\n")
        
        with fs.open(latest_mq_violations, 'r') as f:
            mq_violations_data = json.load(f)
        
        print("MODEL QUALITY VIOLATIONS:")
        print(json.dumps(mq_violations_data, indent=2))
        
        # Save locally
        with open('model_quality_violations.json', 'w') as f:
            json.dump(mq_violations_data, f, indent=2)
        print("\n✅ Saved to: model_quality_violations.json")
        
    else:
        print("⚠️ No model quality results yet - schedule hasn't run")
        print(f"\nCheck back after the next scheduled run (hourly)")
    
    # Find statistics
    mq_statistics_files = [f for f in all_mq_files if 'statistics.json' in f]
    
    if mq_statistics_files:
        latest_mq_stats = mq_statistics_files[-1]
        print(f"\n📊 Latest Statistics: {latest_mq_stats}\n")
        
        with fs.open(latest_mq_stats, 'r') as f:
            mq_stats_data = json.load(f)
        
        print("MODEL QUALITY METRICS:")
        print(json.dumps(mq_stats_data, indent=2))
        
        # Save locally
        with open('model_quality_statistics.json', 'w') as f:
            json.dump(mq_stats_data, f, indent=2)
        print("\n✅ Saved to: model_quality_statistics.json")

except Exception as e:
    print(f"❌ Error: {e}")

DOWNLOADING MONITORING REPORTS

📊 DATA QUALITY REPORTS
❌ Error: S3FileSystem._ls() got an unexpected keyword argument 'recursive'

📈 MODEL QUALITY REPORTS
❌ Error: S3FileSystem._ls() got an unexpected keyword argument 'recursive'


In [52]:
# SUMMARY
print("\n" + "="*70)
print("SUMMARY")
print("="*70)

import os

local_files = [
    'data_quality_violations.json',
    'data_quality_statistics.json',
    'model_quality_violations.json',
    'model_quality_statistics.json'
]

print("\n📁 Downloaded Files:")
for fname in local_files:
    if os.path.exists(fname):
        size = os.path.getsize(fname)
        print(f"  ✅ {fname} ({size:,} bytes)")
    else:
        print(f"  ⚠️  {fname} (not found)")

print("\n💡 Next Steps:")
print("  1. Open the JSON files to review violations")
print("  2. Check statistics.json for data distributions")
print("  3. Wait for model quality schedule to run (hourly)")


SUMMARY

📁 Downloaded Files:
  ⚠️  data_quality_violations.json (not found)
  ⚠️  data_quality_statistics.json (not found)
  ⚠️  model_quality_violations.json (not found)
  ⚠️  model_quality_statistics.json (not found)

💡 Next Steps:
  1. Open the JSON files to review violations
  2. Check statistics.json for data distributions
  3. Wait for model quality schedule to run (hourly)


# Clean Up

In [53]:
#Run all safety
"""

╭──────────────────────────────────────────────────────────────────────────────────────────────────╮
│ """                                                                                              │
│ ▲                                                                                                │
╰──────────────────────────────────────────────────────────────────────────────────────────────────╯
SyntaxError: incomplete input

In [54]:
sagemaker = boto3.client('sagemaker')
cw_client = boto3.client('cloudwatch')

print("="*70)
print("CLEANUP SCRIPT - Removing All SageMaker Resources")
print("="*70)

# 1. Delete monitoring schedules
print("\n1. Deleting Monitoring Schedules...")
monitoring_schedules = [
    "xgb-data-quality-schedule",
    "xgb-model-quality-schedule",
]

for schedule_name in monitoring_schedules:
    try:
        sagemaker.delete_monitoring_schedule(MonitoringScheduleName=schedule_name)
        print(f"  ✅ Deleted: {schedule_name}")
    except sagemaker.exceptions.ResourceNotFound:
        print(f"  ⚠️  Not found: {schedule_name}")
    except Exception as e:
        print(f"  ❌ Error deleting {schedule_name}: {e}")

# 2. Delete endpoint
print("\n2. Deleting Endpoint...")
try:
    sagemaker.delete_endpoint(EndpointName=new_xgb_endpoint_name)
    print(f"  ✅ Deleted endpoint: {new_xgb_endpoint_name}")
except Exception as e:
    print(f"  ❌ Error deleting endpoint: {e}")

# 3. Delete endpoint configuration (optional but recommended)
print("\n3. Deleting Endpoint Configuration...")
try:
    # Get the endpoint config name (usually similar to endpoint name)
    endpoint_config_name = new_xgb_endpoint_name.replace('-endpoint-', '-endpoint-config-')
    sagemaker.delete_endpoint_config(EndpointConfigName=endpoint_config_name)
    print(f"  ✅ Deleted endpoint config: {endpoint_config_name}")
except Exception as e:
    print(f"  ⚠️  Endpoint config deletion: {e}")

# 4. Delete CloudWatch alarms
print("\n4. Deleting CloudWatch Alarms...")
alarms_to_delete = [
    "XGB-Endpoint-High-Latency",
    "XGB-Endpoint-Invocation-Errors",
    "XGB-Endpoint-5XX-Errors-High",
    "XGB-Endpoint-4XX-Errors-High"
]

try:
    cw_client.delete_alarms(AlarmNames=alarms_to_delete)
    for alarm in alarms_to_delete:
        print(f"  ✅ Deleted alarm: {alarm}")
except Exception as e:
    print(f"  ❌ Error deleting alarms: {e}")

# 5. Delete CloudWatch Dashboard (optional)
print("\n5. Deleting CloudWatch Dashboard...")
try:
    cw_client.delete_dashboards(DashboardNames=["SageMaker-ML-Benchmarks"])
    print(f"  ✅ Deleted dashboard: SageMaker-ML-Benchmarks")
except Exception as e:
    print(f"  ⚠️  Dashboard deletion: {e}")

# 6. Verify cleanup
print("\n" + "="*70)
print("VERIFICATION - Checking for remaining resources")
print("="*70)

# Check endpoints
print("\nRemaining endpoints:")
response = sagemaker.list_endpoints()
if response['Endpoints']:
    for endpoint in response['Endpoints']:
        print(f"  - {endpoint['EndpointName']} (Status: {endpoint['EndpointStatus']})")
else:
    print("  ✅ No endpoints found")

# Check monitoring schedules
print("\nRemaining monitoring schedules:")
response = sagemaker.list_monitoring_schedules()
if response['MonitoringScheduleSummaries']:
    for schedule in response['MonitoringScheduleSummaries']:
        print(f"  - {schedule['MonitoringScheduleName']}")
else:
    print("  ✅ No monitoring schedules found")

# Check CloudWatch alarms
print("\nRemaining CloudWatch alarms (XGB-related):")
try:
    response = cw_client.describe_alarms(AlarmNamePrefix="XGB-")
    if response['MetricAlarms']:
        for alarm in response['MetricAlarms']:
            print(f"  - {alarm['AlarmName']}")
    else:
        print("  ✅ No XGB-related alarms found")
except Exception as e:
    print(f"  ⚠️  Error checking alarms: {e}")

print("\n" + "="*70)
print("CLEANUP COMPLETE!")
print("="*70)
print("\nNote: S3 data (models, baselines, monitoring outputs) is NOT deleted.")
print("If you want to delete S3 data, you'll need to manually clean up:")
print(f"  - s3://{bucket}/{prefix}/")
print("\nCosts should now drop to near-zero (only S3 storage costs).")
print("="*70)

CLEANUP SCRIPT - Removing All SageMaker Resources

1. Deleting Monitoring Schedules...
  ✅ Deleted: xgb-data-quality-schedule
  ✅ Deleted: xgb-model-quality-schedule

2. Deleting Endpoint...
  ❌ Error deleting endpoint: An error occurred (ValidationException) when calling the DeleteEndpoint operation: The Endpoint currently has one or more MonitoringSchedules. Please delete the MonitoringSchedules before deleting the Endpoint.

3. Deleting Endpoint Configuration...
  ⚠️  Endpoint config deletion: An error occurred (ValidationException) when calling the DeleteEndpointConfig operation: Could not find endpoint configuration "xgb-benchmark-endpoint-config-20260210-045052".

4. Deleting CloudWatch Alarms...
  ✅ Deleted alarm: XGB-Endpoint-High-Latency
  ✅ Deleted alarm: XGB-Endpoint-Invocation-Errors
  ✅ Deleted alarm: XGB-Endpoint-5XX-Errors-High
  ✅ Deleted alarm: XGB-Endpoint-4XX-Errors-High

5. Deleting CloudWatch Dashboard...
  ✅ Deleted dashboard: SageMaker-ML-Benchmarks

VERIFICATION

In [55]:
print("="*70)
print("RETRY CLEANUP - Deleting Remaining Resources")
print("="*70)

# Wait for monitoring schedule deletions to propagate
print("\nWaiting 30 seconds for monitoring schedule deletions to propagate...")
time.sleep(30)

# 1. Delete monitoring schedules again (in case they weren't fully deleted)
print("\n1. Ensuring Monitoring Schedules are deleted...")
monitoring_schedules = [
    "xgb-data-quality-schedule",
    "xgb-model-quality-schedule",
]

for schedule_name in monitoring_schedules:
    try:
        sagemaker.delete_monitoring_schedule(MonitoringScheduleName=schedule_name)
        print(f"  ✅ Deleted: {schedule_name}")
    except sagemaker.exceptions.ResourceNotFound:
        print(f"  ✅ Already deleted: {schedule_name}")
    except Exception as e:
        print(f"  ❌ Error: {e}")

# Wait again
print("\nWaiting another 30 seconds...")
time.sleep(30)

# 2. Try deleting endpoint again
print("\n2. Deleting Endpoint...")
endpoint_name = "xgb-benchmark-endpoint-20260208-041551"
try:
    sagemaker.delete_endpoint(EndpointName=endpoint_name)
    print(f"  ✅ Deleted endpoint: {endpoint_name}")
except Exception as e:
    print(f"  ❌ Error deleting endpoint: {e}")

# 3. Delete endpoint configuration with correct name
print("\n3. Deleting Endpoint Configuration...")
endpoint_config_name = "xgb-benchmark-endpoint-config-20260208-041551"  
try:
    sagemaker.delete_endpoint_config(EndpointConfigName=endpoint_config_name)
    print(f"  ✅ Deleted endpoint config: {endpoint_config_name}")
except sagemaker.exceptions.ResourceNotFound:
    print(f"  ✅ Endpoint config already deleted or doesn't exist")
except Exception as e:
    print(f"  ❌ Error: {e}")

# 4. Final verification
print("\n" + "="*70)
print("FINAL VERIFICATION")
print("="*70)

print("\nRemaining endpoints:")
response = sagemaker.list_endpoints()
if response['Endpoints']:
    for endpoint in response['Endpoints']:
        print(f"  ⚠️  {endpoint['EndpointName']} (Status: {endpoint['EndpointStatus']})")
else:
    print("  ✅ No endpoints found")

print("\nRemaining monitoring schedules:")
response = sagemaker.list_monitoring_schedules()
if response['MonitoringScheduleSummaries']:
    for schedule in response['MonitoringScheduleSummaries']:
        print(f"  ⚠️  {schedule['MonitoringScheduleName']}")
else:
    print("  ✅ No monitoring schedules found")

print("\n" + "="*70)
print("✅ CLEANUP COMPLETE!")
print("="*70)

RETRY CLEANUP - Deleting Remaining Resources

Waiting 30 seconds for monitoring schedule deletions to propagate...

1. Ensuring Monitoring Schedules are deleted...
  ✅ Already deleted: xgb-data-quality-schedule
  ✅ Already deleted: xgb-model-quality-schedule

Waiting another 30 seconds...

2. Deleting Endpoint...
  ❌ Error deleting endpoint: An error occurred (ValidationException) when calling the DeleteEndpoint operation: Could not find endpoint "xgb-benchmark-endpoint-20260208-041551".

3. Deleting Endpoint Configuration...
  ❌ Error: An error occurred (ValidationException) when calling the DeleteEndpointConfig operation: Could not find endpoint configuration "xgb-benchmark-endpoint-config-20260208-041551".

FINAL VERIFICATION

Remaining endpoints:
  ⚠️  xgb-benchmark-endpoint-20260210-045052 (Status: InService)
  ⚠️  xgb-benchmark-endpoint-20260209-062303 (Status: InService)
  ⚠️  xgb-benchmark-endpoint-20260209-023949 (Status: InService)

Remaining monitoring schedules:
  ✅ No moni